### Libraries.

In [1]:
import math
import os
import os
import sys
from typing import Literal, OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
import scvi
from sklearn.model_selection import train_test_split
import torch
from tqdm import tqdm

from sc_exp_design.config import NeuralVelocityFieldConfig
from sc_exp_design.models import FlowMatching
from sc_exp_design.utils import set_reproducibility

sys.path.insert(0, "../../scripts")
from fetch_data import get_log_transformed_experimental_variables
from data_utils import (
    get_one_hot_encoded_protocols,
    get_one_hot_encoded_protocol_axis
)


### Constants.

In [2]:
# paths
H5AD_PATH = "/Users/lorenzo.consoli/Work/data/collab-goettgens/ds_HID01_logicle_100k_UMAP_ann.h5ad"

# columns
EXPERIMENTAL_COLUMNS = [
    'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]',
    'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]',
    'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'o2_[%]'
]
SCATTER_COLUMNS = ['FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width']
LOG1P_EXP_COL = ["um171_[nm]", "um729_[µm]", "scf_[ng_ml]", 
                 "butyzamide_[nm]", "o2_[%]",
                 "sr1_[nm]", "mtg_[µm]", "rhflt3l_[ng_ml]", "gm-csf_[ng_ml]"]
LOG21P_EXP_COL = ["ldl_[ng_ml]", "il3_[ng_ml]", "retinoic_acid_[µm]",]

# adata keys
X_CHANNEL_OBSM_KEY = "X_channel"
OHE_PROTOCOLS_OBS_KEY = "protocol"
OHE_PROTOCOLS_UNS_KEY = "protocol_ohe"
CONCAT_FEATS_OBSM_KEY_SUFFIX = "scatter_concat"
CONDITION_CONCAT_OBSM_KEY = "protocol_axes_concat"

# transformations options
STANDARDIZE_SCATTER_FEATURES = True
STANDARDIZE_CHANNEL_FEATURES = True
STANDARDIZE_PCs = False

# training
TRAIN_BATCH_SIZE = 256
VAL_BATCH_SIZE = 64
LR = 1e-3
NUM_GRAD_STEPS = 50_000
GRAD_STEPS_LOG_INTERVAL = 10
NUM_GRAD_ACCUMULATIONS_STEPS=1

# model
FLOW_TYPE = "rectified"
COUPLING_TYPE = "independent"
GENERATE_FROM_NOISE = True

# vf
ENCODE_STATE = True
ENCODE_TIME = True
USE_SINUSOIDAL_TIME_FEATURES = True
ENCODE_CONDITIONS = True

STATE_LATENT_DIM = 16
TIME_LATENT_DIM = 4
CONDITION_LATENT_DIM = 4


STATE_ENCODER_HIDDEN_DIMS = (128, 128)
VF_DECODER_HIDDEN_DIMS = (128, 128)
CONDITION_ENCODER_HIDDEN_DIMS = (64, 64)
TIME_ENCODER_HIDDEN_DIMS = (64, 64)

DEVICE = "mps"

# sampling
N_SAMPLES = 5_000

# reproducibility
SEED = 42
set_reproducibility(SEED)


### Reading data.

In [3]:
adata = sc.read_h5ad(H5AD_PATH)
adata

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'positives', 'raw'
    

### Annotating conditions.

In [4]:
# concatenated experimental variable
adata = get_log_transformed_experimental_variables(
    adata,
    EXPERIMENTAL_COLUMNS,
    LOG1P_EXP_COL,
    LOG21P_EXP_COL,
    CONDITION_CONCAT_OBSM_KEY
)

# one hot encoding of concatenated protocol axes
adata = get_one_hot_encoded_protocols(
    adata,
    EXPERIMENTAL_COLUMNS,
    OHE_PROTOCOLS_OBS_KEY,
    OHE_PROTOCOLS_UNS_KEY
)


# one hot encoding of concatenated protocol axes
adata = get_one_hot_encoded_protocol_axis(
    adata,
    EXPERIMENTAL_COLUMNS,
    OHE_PROTOCOLS_UNS_KEY
)
adata


AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type', 'protocol'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap', 'protocol_ohe', 'mtg_[µm]_protocol_ohe', 'rhflt3l_

### Splitting data.

In [5]:
train_adata, val_adata = train_test_split(adata, test_size=0.3)
train_adata.shape, val_adata.shape

((70000, 21), (30000, 21))

### Tranforming data.

In [6]:
# retrieving scatter features and optionally normalizing them
X_scatter_train = train_adata.obs[SCATTER_COLUMNS].values
X_scatter_val =  val_adata.obs[SCATTER_COLUMNS].values
X_scatter_mean = X_scatter_train.mean(0)
X_scatter_std = X_scatter_train.std(0)
if STANDARDIZE_SCATTER_FEATURES:
    print("Standardizing scatter features")
    X_scatter_train = (X_scatter_train - X_scatter_mean)/X_scatter_std
    X_scatter_val = (X_scatter_val - X_scatter_mean)/X_scatter_std
assert np.isfinite(X_scatter_train).all()
assert np.isfinite(X_scatter_val).all()


# concatenating with channel features
train_adata.obsm[X_CHANNEL_OBSM_KEY] = train_adata.X
val_adata.obsm[X_CHANNEL_OBSM_KEY] = val_adata.X
X_channel_train = train_adata.X
X_channel_val = val_adata.X
X_channel_mean = X_channel_train.mean(0)
X_channel_std = X_channel_train.std(0)
if STANDARDIZE_CHANNEL_FEATURES:
    print("Standardizing channel features")
    X_channel_train = (X_channel_train - X_channel_mean)/X_channel_std
    X_channel_val = (X_channel_val - X_channel_mean)/X_channel_std
assert np.isfinite(X_channel_train).all()
assert np.isfinite(X_channel_val).all()

train_adata.X = X_channel_train
val_adata.X = X_channel_val
train_adata.obsm[f"X_channel_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_train, X_scatter_train), axis=1)
val_adata.obsm[f"X_channel_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_val, X_scatter_val), axis=1)

# concatenating with PCs
X_pca_train = train_adata.obsm["X_pca"]
X_pca_val = val_adata.obsm["X_pca"]
X_pca_mean = X_pca_train.mean(0)
X_pca_std = X_pca_train.std(0)
if STANDARDIZE_PCs:
    print("Standardizing PCs")
    X_pca_train = (X_pca_train - X_pca_mean)/X_pca_std
    X_pca_val = (X_pca_val - X_pca_mean)/X_pca_std
assert np.isfinite(X_pca_train).all()
assert np.isfinite(X_pca_val).all()

train_adata.obsm["X_pca"] = X_pca_train
val_adata.obsm["X_pca"] = X_pca_val
train_adata.obsm[f"X_pca_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_pca_train, X_scatter_train), axis=1)
val_adata.obsm[f"X_pca_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_pca_val, X_scatter_val), axis=1)
train_adata, val_adata

Standardizing scatter features
Standardizing channel features


/var/folders/p4/lyzkxmgs697cq8dyh_h0yh0c0000gn/T/ipykernel_52808/741337475.py:15: ImplicitModificationWarning: Setting element `.obsm['X_channel']` of view, initializing view as actual.
  train_adata.obsm[X_CHANNEL_OBSM_KEY] = train_adata.X
/var/folders/p4/lyzkxmgs697cq8dyh_h0yh0c0000gn/T/ipykernel_52808/741337475.py:16: ImplicitModificationWarning: Setting element `.obsm['X_channel']` of view, initializing view as actual.
  val_adata.obsm[X_CHANNEL_OBSM_KEY] = val_adata.X


(AnnData object with n_obs × n_vars = 70000 × 21
     obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type', 'protocol'
     var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
     uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap', 'protocol_ohe', 'mtg_[µm]_protocol_ohe', 'rhflt

### Predefining nn settings.

In [7]:
get_mlp_config = lambda hidden_dims, final_activation_class=torch.nn.Identity: {
    "hidden_dims": hidden_dims,
    "use_batchnorm": False,
    "use_dropout": False,
    "activation_class": torch.nn.ReLU,
    "final_activation_class": final_activation_class,
}
get_lm_config = lambda final_activation_class=torch.nn.Identity: get_mlp_config((), final_activation_class=final_activation_class)

get_cond_encoder_concat = lambda hidden_dims=CONDITION_ENCODER_HIDDEN_DIMS: {
    "perturbation_layers_before_pooling":{
        f"feats_{CONDITION_CONCAT_OBSM_KEY}_{CONDITION_CONCAT_OBSM_KEY}": 
        {
            "input_dim": adata.obsm[CONDITION_CONCAT_OBSM_KEY].shape[-1],
            "output_dim": CONDITION_LATENT_DIM,
            **get_mlp_config(
                hidden_dims
            )
        }
    },
    "perturbation_layers_after_pooling": get_lm_config()
}
get_cond_encoder_ohe = lambda hidden_dims=CONDITION_ENCODER_HIDDEN_DIMS: {
    "perturbation_layers_before_pooling":{
        f"repr_{OHE_PROTOCOLS_OBS_KEY}_{OHE_PROTOCOLS_UNS_KEY}": {
            "input_dim": list(adata.uns[OHE_PROTOCOLS_UNS_KEY].values())[0].shape[-1],
            "output_dim": CONDITION_LATENT_DIM,
            **get_mlp_config(hidden_dims)
        }
    },
    "perturbation_layers_after_pooling": get_lm_config()
}
get_cond_encoder_ohe_axis = lambda hidden_dims=CONDITION_ENCODER_HIDDEN_DIMS: {
    "perturbation_layers_before_pooling":{
        f"repr_{protocol_axis}_{protocol_axis}_{protocol_axis}_{OHE_PROTOCOLS_UNS_KEY}":  {
            "input_dim": list(adata.uns[f"{protocol_axis}_{OHE_PROTOCOLS_UNS_KEY}"].values())[0].shape[-1],
            "output_dim": CONDITION_LATENT_DIM,
            **get_mlp_config(hidden_dims)
        }
        for protocol_axis in EXPERIMENTAL_COLUMNS
    },
    "perturbation_layers_after_pooling": get_lm_config()
}

### Defining data settings.

In [8]:
# defining dictionary for each conditioning mode settings
conditioning_mode_data_setting_dict = {
    #"unconditional": {}, 
    "protocol_concat": {
        "perturbations": (CONDITION_CONCAT_OBSM_KEY),
        "perturbations_in_obsm": (CONDITION_CONCAT_OBSM_KEY),
        "perturbation_reps": {CONDITION_CONCAT_OBSM_KEY: CONDITION_CONCAT_OBSM_KEY},
    },
    #"protocol_ohe": {
    #    "perturbations": (OHE_PROTOCOLS_OBS_KEY, ),
    #    "perturbation_reps": {OHE_PROTOCOLS_OBS_KEY: OHE_PROTOCOLS_UNS_KEY}
    #},
    #"protocol_axes_ohe": {
    #    "perturbations": EXPERIMENTAL_COLUMNS,
    #    "perturbation_reps": {protocol_ax: f"{protocol_ax}_{OHE_PROTOCOLS_UNS_KEY}" for protocol_ax in EXPERIMENTAL_COLUMNS}
    #}
}
conditioning_mode_setting_dict = {
    "unconditional": lambda: {}, 
    "protocol_concat": get_cond_encoder_concat,
    #"protocol_ohe": get_cond_encoder_ohe,
    #"protocol_axes_ohe": get_cond_encoder_ohe_axis,
}

#reprs = [X_CHANNEL_OBSM_KEY, "X_pca", f"X_channel_{CONCAT_FEATS_OBSM_KEY_SUFFIX}", f"X_pca_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"]
reprs = [X_CHANNEL_OBSM_KEY, f"X_channel_{CONCAT_FEATS_OBSM_KEY_SUFFIX}",]

# initializing data managers
counter = 0
model_dict = {}
for repr in reprs:
    for conditioning_mode, conditioning_settings in conditioning_mode_data_setting_dict.items():
        # info message
        model_id = f"{repr}_{conditioning_mode}"
        print(f"Fitting model {model_id} ({counter}/{len(conditioning_mode_data_setting_dict)*len(reprs)})")
        if model_id in model_dict:
            continue
        #if ("channel" not in model_id) or ("protocol_concat" not in model_id):
        #    continue

        # initializing model and preparing data
        model = FlowMatching(
            flow_type=FLOW_TYPE,
            coupling_type=COUPLING_TYPE,
            generate_from_noise=GENERATE_FROM_NOISE,
            device_id=DEVICE,
        )
        model.prepare_train_data(
            train_adata,
            repr,
            **conditioning_settings
        )

        # initializing config velocity field
        cond_settings = conditioning_mode_setting_dict[conditioning_mode]()
        optimizer_class = torch.optim.Adam

        vf_config = NeuralVelocityFieldConfig(
            model.train_data.state_data.shape[-1],
            encode_state=ENCODE_STATE,
            state_encoder_output_dim=STATE_LATENT_DIM,
            state_encoder_mlp_kwargs=get_mlp_config(STATE_ENCODER_HIDDEN_DIMS),
            encode_time=ENCODE_TIME,
            use_sinusoidal_time_features=USE_SINUSOIDAL_TIME_FEATURES,
            time_encoder_output_dim=TIME_LATENT_DIM,
            time_encoder_mlp_kwargs=get_mlp_config(TIME_ENCODER_HIDDEN_DIMS),
            use_guidance=len(conditioning_settings) > 0,
            encode_conditions=ENCODE_CONDITIONS,
            perturbation_encoder_output_dim=CONDITION_LATENT_DIM,
            perturbation_layers_before_pooling=cond_settings.get("perturbation_layers_before_pooling", None),
            perturbation_layers_after_pooling=cond_settings.get("perturbation_layers_after_pooling", None),
            decoder_mlp_kwargs=get_mlp_config(VF_DECODER_HIDDEN_DIMS),
        )
        model.prepare_model(
            vf_config,
            optimizer_class=optimizer_class,
            optimizer_kwargs={"lr": LR}
        )

        model.train(
            num_training_steps=NUM_GRAD_STEPS,
            train_batch_size=TRAIN_BATCH_SIZE,
            grad_steps_log_interval=GRAD_STEPS_LOG_INTERVAL,
            num_grad_accumulation_steps=NUM_GRAD_ACCUMULATIONS_STEPS,
        )
        model_dict[model_id] = model
        counter += 1

/Users/lorenzo.consoli/micromamba/envs/sc-exp-design/lib/python3.10/site-packages/torch/_compile.py:31: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


Fitting model X_channel_protocol_concat (0/2)


Loss: 0.1469: 100%|██████████| 50000/50000 [10:23<00:00, 80.20it/s] 


Fitting model X_channel_scatter_concat_protocol_concat (1/2)


Loss: 1.2855: 100%|██████████| 50000/50000 [07:36<00:00, 109.43it/s]
